# CartoonStudio — Kaggle 3D story renderer
**Rigged-character preview:** Blender Cycles, Ara 3D Gen human models, story-based action/emotion cues, Rhubarb phoneme lip-sync and Piper dialogue. Internet and an NVIDIA GPU are needed for first-time setup. The default output is an 8-second 1280×720 preview.

This notebook includes the CartoonStudio project source. You do not need to attach a ZIP or dataset. It can generate detailed rigged human characters; polished feature-film quality still depends on the imported model, usable animation clips, and a suitable staged environment.


In [ ]:
# Extract the project source embedded in this notebook (no Kaggle input dataset required).
from pathlib import Path
import base64, hashlib, zipfile, io, subprocess, sys

ROOT = Path('/kaggle/working/CartoonStudio')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = '...'
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
expected_sha256 = '4ef658171e57c59411f149b190b720ef2a765c80b57c5422f53fbce75edc3db8'
if hashlib.sha256(archive_bytes).hexdigest() != expected_sha256:
    raise RuntimeError('Embedded CartoonStudio source archive checksum mismatch.')
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for name in z.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError(f'Unsafe path in embedded project archive: {name}')
    z.extractall('/kaggle/working')
print('Embedded source extracted:', ROOT)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'No NVIDIA GPU detected')


In [ ]:
# Install Blender, FFmpeg, Piper and a compatible Node runtime for Ara 3D Gen.
import subprocess, sys, shutil, os, platform, tarfile, io, requests
base_tools = ('ffmpeg', 'ffprobe', 'espeak', 'git', 'node', 'npm')
if not shutil.which('blender') or any(not shutil.which(tool) for tool in base_tools):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'blender', 'ffmpeg', 'espeak', 'git', 'nodejs', 'npm', 'xz-utils'], check=True)
if not shutil.which('blender'):
    raise RuntimeError('Blender installation failed: executable not found after apt install.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'piper-tts', 'requests'], check=True)
node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    # Kaggle images may ship Node 18. Install the current Node 22 LTS binary into
    # the session and put it first on PATH so npm uses it too.
    arch = {'x86_64':'x64', 'aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise RuntimeError(f'Unsupported Kaggle CPU architecture for Node: {platform.machine()}')
    releases = requests.get('https://nodejs.org/dist/index.json', timeout=30)
    releases.raise_for_status()
    release = next(item for item in releases.json() if item['version'].startswith('v22.') and f'linux-{arch}' in item['files'])
    version = release['version']
    archive_url = f'https://nodejs.org/dist/{version}/node-{version}-linux-{arch}.tar.xz'
    response = requests.get(archive_url, timeout=180)
    response.raise_for_status()
    node_root = Path('/kaggle/working/node22')
    node_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(response.content), mode='r:xz') as tf:
        base = f'node-{version}-linux-{arch}/'
        members = [m for m in tf.getmembers() if m.name.startswith(base)]
        for member in members:
            relative = Path(member.name[len(base):])
            target = (node_root / relative).resolve()
            if not target.is_relative_to(node_root.resolve()):
                raise RuntimeError('Unsafe path in Node.js release archive')
        tf.extractall(node_root, members=members, filter='data')
    os.environ['PATH'] = str(node_root/'bin') + os.pathsep + os.environ['PATH']
    node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    raise RuntimeError(f'Node 20+ is required by Ara 3D Gen; found {node_major}.')
print('Blender:', subprocess.check_output(['blender','--version'],text=True).splitlines()[0])
print('FFmpeg:', subprocess.check_output(['ffmpeg','-version'],text=True).splitlines()[0])
print('Node:', subprocess.check_output(['node','--version'],text=True).strip())


In [ ]:
# Fail BEFORE the expensive video job if CUDA is not usable by Blender.
preflight = subprocess.run(['blender','-b','--python',str(ROOT/'renderer'/'check_cuda.py')], capture_output=True, text=True)
print(preflight.stdout[-3500:])
print(preflight.stderr[-1200:])
if preflight.returncode != 0:
    raise RuntimeError('Blender Cycles CUDA unavailable in this Kaggle environment. Do not continue to the long GPU render. Switch renderer or compute environment.')

In [ ]:
# Install free character generation and phoneme lip-sync tools (one-time download per Kaggle session).
import json, os, requests, zipfile
from pathlib import Path

ARA_ROOT = Path('/kaggle/working/ara-3d-gen')
if not (ARA_ROOT/'package.json').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/olatunde85/3d-gen.git',str(ARA_ROOT)],check=True)
node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    raise RuntimeError('Ara 3D Gen requires Node 20+. Use a current Kaggle image with Node 20 or newer.')
if not (ARA_ROOT/'node_modules').exists():
    subprocess.run(['npm','ci'],cwd=ARA_ROOT,check=True)
if not (ARA_ROOT/'public'/'assets'/'catalog.json').exists():
    subprocess.run(['npm','run','setup-assets'],cwd=ARA_ROOT,check=True)

RHU_DIR = Path('/kaggle/working/rhubarb')
RHU_DIR.mkdir(parents=True,exist_ok=True)
rhubarb_binary = next(RHU_DIR.rglob('rhubarb'), None)
if rhubarb_binary is None:
    release = requests.get('https://api.github.com/repos/DanielSWolf/rhubarb-lip-sync/releases/latest',timeout=30)
    release.raise_for_status()
    asset = next((a for a in release.json()['assets'] if 'Linux.zip' in a['name']), None)
    if asset is None:
        raise RuntimeError('Could not find the official Rhubarb Linux release asset.')
    archive = RHU_DIR/'rhubarb-linux.zip'
    response = requests.get(asset['browser_download_url'],timeout=120)
    response.raise_for_status()
    archive.write_bytes(response.content)
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target=(RHU_DIR/member.filename).resolve()
            if not target.is_relative_to(RHU_DIR.resolve()):
                raise ValueError('Unsafe path in Rhubarb release archive')
        zf.extractall(RHU_DIR)
    rhubarb_binary = next(RHU_DIR.rglob('rhubarb'), None)
if rhubarb_binary is None:
    raise FileNotFoundError('Rhubarb release unpacked without a rhubarb executable.')
rhubarb_binary.chmod(0o755)
print('Rigged human generator ready:', ARA_ROOT)
print('Phoneme lip-sync ready:', rhubarb_binary)


In [ ]:
# Download voice models AFTER preflight. Internet must be enabled in notebook settings.
voices = ROOT/'voices'
voices.mkdir(exist_ok=True)
for name in ('en_GB-alan-medium', 'en_US-amy-medium'):
    if not (voices/(name+'.onnx')).exists():
        subprocess.run([sys.executable,'-m','piper.download_voices','--data-dir',str(voices),name],check=True)
print('Voice models ready:', [p.name for p in voices.glob('*.onnx')])

In [ ]:
# Generate a 3D talking-character MP4. This runs only when you execute this cell.
import time
OUT = Path('/kaggle/working/cartoon_studio_output')
OUT.mkdir(exist_ok=True)
cmd = [sys.executable, '-m', 'cartoon_studio.cli', 'generate',
       '--story', str(ROOT/'stories'/'first_story.json'), '--voices', str(voices),
       '--work', str(OUT/'work'), '--output', str(OUT/'first_cartoon.mp4'),
       '--ara-root', str(ARA_ROOT), '--style', 'auto', '--quality', 'high',
       '--rhubarb', str(rhubarb_binary), '--strict-gpu', '--tts', 'piper', '--renderer', 'cycles',
       '--width', '1280', '--height', '720', '--fps', '24', '--samples', '32',
       '--preview-seconds', '8']
t0=time.monotonic()
subprocess.run(cmd, check=True, cwd=ROOT)
print('Video output:', OUT/'first_cartoon.mp4')
print('Total elapsed seconds:', round(time.monotonic()-t0, 1))
from IPython.display import display, Video
display(Video(str(OUT/'first_cartoon.mp4'), embed=True))


In [ ]:
# Estimate weekly production using the ACTUAL measured rendering speed from above.
import json
stats = json.loads((OUT/'work'/'render_log.json').read_text())
spf = stats['seconds_per_frame']
subprocess.run([sys.executable,'-m','cartoon_studio.budget','--minutes','20',
                '--seconds-per-frame',str(spf),'--weekly-hours','30','--reserve','0.15'],
               cwd=ROOT,check=True)
print('Measured seconds per frame INCLUDING Blender launch:', round(spf, 3))
print('This is a short-render estimate, not a 20-minute validation.')

## Creating the full story
Review the 8-second preview first. To render the full story, change `--preview-seconds 8` to `--preview-seconds 0` in the generation cell. Keep 1280×720 and 24 FPS for the quality preset, or reduce to 640×360 with 8 samples to conserve Kaggle GPU time. The generated characters, animation choices, speech cues and scene preset are driven by `stories/first_story.json`.

The Ara generator creates high-detail rigged people with facial blend shapes and motion clips; Rhubarb maps speech to those face shapes. `visual_style: auto` selects flatter stylized cartoon materials or natural human materials from the story wording. The renderer still has lightweight procedural sets; add licensed scene assets or an HDRI for a production environment. No output is guaranteed until you inspect a rendered MP4.